# Part A — Waves in a box

**Working time:** one 90-minute block
**Work in groups of 2–4.** This notebook is guided and is not submitted.

## Learning goals

By the end you should be able to:

- configure and run the shallow-water model;
- predict and measure the long-wave speed $c=\sqrt{gH}$;
- identify an incident and reflected wave;
- carry out a controlled comparison in which only one parameter changes.

Start with the animation below: watch first, then use the Hovmöller
diagram and a speed measurement to explain what you saw.


**Prediction 1.** What will a localized elevation do after the model starts?

**Solution.** It launches gravity waves that spread away from the initial bump. They reflect from the four solid walls and cross the basin again.


## 1. Environment check and imports

Before the lab, follow `INSTALLATION.md` to create the `shallowwater-lab`
Miniconda environment and select it as the kernel in VS Code. The package
command used there is:

```text
python -m pip install "shallowwater==0.1.4"
```

Optional acceleration is installed with:

```text
python -m pip install "shallowwater[numba]==0.1.4"
```

Numba is optional. Its first run can pause while functions are compiled.
Ask for help now if the import cell fails; do not change model code to
repair an installation problem.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from shallowwater import (
    ModelParams, animate_eta, backend_info, compute_dt_cfl, depth_on_u,
    make_grid, run_model, setup_initial_state, zero_forcing,
)

print(backend_info())


## 2. First run: watch a wave spread and reflect

The initial state is a circular bump in surface elevation with zero
velocity. Run the next two cells. The animation appears directly below
the cell and is also saved as a GIF in the course `animations/` folder.

The helper deliberately shows only about 40 evenly spaced frames. You
can reuse it later with any model output that contains `eta`.


In [ ]:
def find_course_root():
    candidates = (
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "MT1562_python_lab_waves",
    )
    return next(path for path in candidates if (path / "notebooks").exists())


COURSE_ROOT = find_course_root()
ANIMATION_DIR = COURSE_ROOT / "animations"
ANIMATION_DIR.mkdir(exist_ok=True)


def animate_and_save(out, grid, filename, *, title, max_frames=42):
    # Display a compact eta animation and save the same frames as a GIF.
    frame_count = len(out["time"])
    frames = np.unique(
        np.linspace(0, frame_count - 1, min(max_frames, frame_count), dtype=int)
    )
    animation = animate_eta(
        out, grid, frames=frames, interval=100, repeat=True,
        title=title, contours=False, remove_mean=False,
        figsize=(8.5, 3.6),
    )
    path = ANIMATION_DIR / filename
    animation.save(str(path), fps=10, dpi=85)
    print("Saved animation:", path.resolve())
    return animation


In [ ]:
visual_grid = make_grid(96, 56, 1.2e6, 700e3)
visual_params = ModelParams(
    H=500.0, g=9.81, f0=0.0, beta=0.0, r=0.0, linear=True,
)
visual_dt = compute_dt_cfl(visual_grid, visual_params, cfl=0.45)
visual_out = run_model(
    tmax=5.0*3600, dt=visual_dt,
    grid=visual_grid, params=visual_params,
    forcing_fn=zero_forcing,
    ic_fn=lambda g, p: setup_initial_state(
        g, p, mode="gaussian_bump", amp=0.12, R=70e3,
        x0=0.35*g.Lx, y0=0.55*g.Ly,
    ),
    save_every=4, out_vars=("eta",),
)

first_wave_animation = animate_and_save(
    visual_out, visual_grid, "part_a_first_wave.gif",
    title="Circular shallow-water wave: propagation and reflection",
)
first_wave_animation


**Observation 1.** Describe two things you saw before and after the first wall reflection.

**Solution.** Before reflection, the initially circular disturbance expands outward. The nearest walls are reached first; reflected wave fronts then reverse their normal direction and interfere with waves arriving from other walls.


## 3. A controlled right-going pulse

The model variables are surface displacement $\eta$ and depth-averaged
velocities $(u,v)$. The initial velocity below is chosen so most of the
disturbance travels toward increasing $x$, which makes speed and
reflection easier to measure.

The model uses solid vertical walls. It does not include wave breaking,
wetting and drying, or coastal inundation.


In [ ]:
def cross_basin_pulse(grid, params, *, amplitude=0.10, radius=60e3, x0=300e3):
    eta_line = amplitude * np.exp(-((grid.x_c - x0) / radius) ** 2)
    eta = np.repeat(eta_line[None, :], grid.Ny, axis=0)

    H_u = depth_on_u(grid, params.H)
    eta_u = amplitude * np.exp(-((grid.x_u - x0) / radius) ** 2)
    u = np.repeat(eta_u[None, :], grid.Ny, axis=0) * np.sqrt(params.g / H_u)
    v = np.zeros((grid.Ny + 1, grid.Nx))
    return eta, u, v


def run_uniform_case(*, H=400.0, Lx=1.6e6, Nx=160, tmax_hours=8.0):
    Ny, Ly = 20, 200e3
    grid = make_grid(Nx, Ny, Lx, Ly)
    params = ModelParams(H=H, g=9.81, f0=0.0, beta=0.0, r=0.0, linear=True)
    dt = compute_dt_cfl(grid, params, cfl=0.45)
    out = run_model(
        tmax=tmax_hours * 3600,
        dt=dt,
        grid=grid,
        params=params,
        forcing_fn=zero_forcing,
        ic_fn=lambda g, p: cross_basin_pulse(g, p),
        save_every=4,
        out_vars=("eta", "u"),
    )
    print(
        f"H={H:.0f} m, Lx={Lx/1e3:.0f} km, dx={grid.dx/1e3:.1f} km, "
        f"dt={dt:.1f} s, saved={len(out['time'])}"
    )
    return grid, params, out


In [ ]:
grid, params, out = run_uniform_case()
eta = np.asarray(out["eta"])
times = np.asarray(out["time"])
eta_line = eta.mean(axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
image = ax.pcolormesh(grid.x_c / 1e3, times / 3600, eta_line, shading="auto", cmap="RdBu_r")
ax.set(xlabel="x [km]", ylabel="time [hours]", title="Centreline Hovmöller diagram")
fig.colorbar(image, ax=ax, label="surface displacement [m]")
plt.show()


In [ ]:
uniform_animation = animate_and_save(
    out, grid, "part_a_uniform_400m.gif",
    title="Right-going pulse and reflection: H = 400 m",
)
uniform_animation


**Observation 2.** Identify the incident and reflected branches in the Hovmöller diagram. What happens at the eastern wall?

**Solution.** The first diagonal branch moves toward increasing x and is incident on the eastern wall. After landfall, a branch with the opposite slope moves westward. Surface elevation reflects with the same sign at a rigid wall, while the normal velocity reverses.


## 4. Predict and measure wave speed

For a uniform-depth shallow-water wave,

$$
c_{theory}=\sqrt{gH}.
$$

We measure the position of the maximum before the pulse reaches the wall.
A grid introduces uncertainty of roughly one grid cell in position.


In [ ]:
c_theory = np.sqrt(params.g * float(params.H))
target_time = 2.5 * 3600
time_index = int(np.argmin(np.abs(times - target_time)))
x_initial = 300e3
x_peak = grid.x_c[np.argmax(eta_line[time_index])]
c_measured = (x_peak - x_initial) / times[time_index]
relative_error = abs(c_measured - c_theory) / c_theory

print(f"theoretical speed = {c_theory:.2f} m/s")
print(f"measured speed    = {c_measured:.2f} m/s")
print(f"relative error    = {100*relative_error:.1f} %")


**Analysis 1.** Report the theoretical and measured speeds. Are they consistent given the grid spacing?

**Solution.** The theoretical speed for $H=400$ m is 62.64 m/s. The measured value should be close, normally within a few percent. A one-cell position uncertainty is 10 km, already about 1.8% of the distance travelled in 2.5 hours.


## 5. Controlled depth experiment

Change only the depth from 400 m to 900 m. Predict the speed ratio before
running. Keep the domain and initial disturbance unchanged.


**Prediction 2.** What is $c_{900}/c_{400}$? Will reflection occur earlier or later?

**Solution.** The ratio is $\sqrt{900/400}=1.5$. The deeper case is faster, so it reaches and reflects from the eastern wall earlier.


In [ ]:
grid_deep, params_deep, out_deep = run_uniform_case(H=900.0, tmax_hours=5.5)
eta_deep = np.asarray(out_deep["eta"]).mean(axis=1)
times_deep = np.asarray(out_deep["time"])

fig, ax = plt.subplots(figsize=(9, 4))
image = ax.pcolormesh(
    grid_deep.x_c / 1e3, times_deep / 3600, eta_deep,
    shading="auto", cmap="RdBu_r"
)
ax.set(xlabel="x [km]", ylabel="time [hours]", title="H = 900 m")
fig.colorbar(image, ax=ax, label="surface displacement [m]")
plt.show()


In [ ]:
deep_animation = animate_and_save(
    out_deep, grid_deep, "part_a_uniform_900m.gif",
    title="Right-going pulse and reflection: H = 900 m",
)
deep_animation


**Analysis 2.** Does the numerical comparison support the predicted depth scaling? Give evidence from the plot or a measurement.

**Solution.** Yes. The incident branch is about 1.5 times steeper in x-versus-time coordinates, and the reflection occurs earlier. A measured speed should be close to 94.0 m/s, compared with 62.6 m/s in the baseline.


## 6. Optional investigations

If time permits, try one of these while changing only one primary factor:

- Double the basin length. How does the wall-arrival time change?
- Move the initial pulse. Which arrival times change and which speeds do not?
- Change the pulse amplitude. Does speed change in the linear model?
- Change `Nx` while keeping `Lx` fixed. How does numerical error change?

## Checkpoint

Save a short record of the parameter changed, prediction, observation,
quantitative evidence, and one model limitation.


**Final reflection.** Name one physical conclusion and one limitation of this experiment.

**Solution.** Physical conclusion: long waves propagate faster in deeper water and reflect from a solid coast. Limitation: this is a linear, hydrostatic, depth-averaged model with permanently wet cells, so it cannot represent breaking or inundation.
